# 02 - Reproduce the USSD audio teacher from raw DAIC-WOZ

Rebuilds participant-only audio from DAIC-WOZ transcripts, extracts ComParE16 with **openSMILE 3.0.2**, audits the frozen USSD run-4 checkpoint on DEV-34, and exports TRAIN-107 KD targets.

Deterministic ComParE16 preprocessing is stored in a persistent shared cache and reused across fresh experiment runs after strict provenance/count validation. It explicitly restores the SMILExtract executable bit after ZIP extraction and prints the complete child-process error if preprocessing fails.

**Selection policy:** teacher epoch 0 is the original released USSD run-4 checkpoint. Fine-tuned epochs may replace it only if they strictly improve the fixed DEV selection key. Otherwise the original epoch-0 teacher remains selected.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, shutil, subprocess, base64

DAIC = Path('/content/drive/MyDrive/DAIC_WOZ')
EXP_ROOT = DAIC / 'experiments' / 'exact_v3_reproduction'
ACTIVE = EXP_ROOT / 'ACTIVE_RUN.txt'
assert ACTIVE.is_file(), 'Run notebooks/00_start_new_run.ipynb first.'
RUN = EXP_ROOT / ACTIVE.read_text().strip()
assert RUN.is_dir(), RUN

REPO = 'engrkhubabahmad/reliability-aware-depression-kd'
CORE_COMMIT = '0d6ac621be7da1db7c1d769d075d142d0a740fd8'
CODE = Path('/content/reliability-aware-depression-kd-exact-v3')

token = userdata.get('GITHUB_TOKEN')
assert token, 'Add GITHUB_TOKEN to Colab Secrets.'
auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
env = os.environ.copy()
env['GIT_CONFIG_COUNT']='1'
env['GIT_CONFIG_KEY_0']='http.https://github.com/.extraheader'
env['GIT_CONFIG_VALUE_0']=f'AUTHORIZATION: basic {auth}'

if CODE.exists(): shutil.rmtree(CODE)
CODE.mkdir(parents=True)
subprocess.run(['git','-C',str(CODE),'init'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'remote','add','origin',f'https://github.com/{REPO}.git'],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'fetch','--depth','1','origin',CORE_COMMIT],check=True,env=env)
subprocess.run(['git','-C',str(CODE),'checkout','--detach','FETCH_HEAD'],check=True,env=env)
head=subprocess.check_output(['git','-C',str(CODE),'rev-parse','HEAD'],text=True).strip()
assert head==CORE_COMMIT,(head,CORE_COMMIT)
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(CODE/'requirements.txt')],check=True)
print('RUN:',RUN)
print('Pinned core:',head)


In [ ]:
import urllib.request, zipfile, hashlib, stat

def sha256(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()

OPENSMILE=Path('/content/opensmile-3.0.2-linux-x86_64')
if not OPENSMILE.exists():
    z=Path('/content/opensmile-3.0.2-linux-x86_64.zip')
    url='https://github.com/audeering/opensmile/releases/download/v3.0.2/opensmile-3.0.2-linux-x86_64.zip'
    urllib.request.urlretrieve(url,z)
    with zipfile.ZipFile(z) as f: f.extractall('/content')

SMILE=OPENSMILE/'bin/SMILExtract'
CONFIG=OPENSMILE/'config/compare16/ComParE_2016.conf'
assert SMILE.is_file() and CONFIG.is_file()

# Python zipfile extraction does not reliably preserve Unix executable bits.
# Make the pinned openSMILE binary executable explicitly on every run.
SMILE.chmod(SMILE.stat().st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)
assert os.access(SMILE,os.X_OK),f'SMILExtract is not executable: {SMILE}'

config_sha=sha256(CONFIG)
assert config_sha=='4baf8b75324db30e632a2935ab469058d54cd1c7b1806d5cc220dc7b124c454c'

# Smoke-test the binary before starting 141 participants.
smoke=subprocess.run([str(SMILE),'-h'],capture_output=True,text=True)
print('SMILExtract executable:',SMILE)
print('SMILExtract return code:',smoke.returncode)
print('SMILExtract banner:',(smoke.stdout or smoke.stderr).splitlines()[:3])
assert smoke.returncode==0,(
    'SMILExtract smoke test failed.\nSTDOUT:\n'+smoke.stdout+'\nSTDERR:\n'+smoke.stderr
)
print('ComParE config SHA256:',config_sha)

In [ ]:
USSD_REPO='https://github.com/vijaysumaravi/USSD-depression.git'
USSD_COMMIT='c3e68649153004ed2174878a1c54c716ad26cfd7'
SRC=Path('/content/USSD-depression')
if SRC.exists(): shutil.rmtree(SRC)
subprocess.run(['git','clone',USSD_REPO,str(SRC)],check=True)
subprocess.run(['git','-C',str(SRC),'checkout','--detach',USSD_COMMIT],check=True)
subprocess.run([sys.executable,'-m','scripts.teachers.ussd_audio.bootstrap','--root',str(SRC)],cwd=CODE,check=True)

In [ ]:
import pandas as pd

# Persistent deterministic ComParE16 cache.
# A new experiment run gets new MODEL/RESULT folders, but raw->ComParE16
# preprocessing is reused when its provenance matches exactly.

CANONICAL_CACHE=DAIC/'experiments'/'ussd_compare16'
SHARED_CACHE=EXP_ROOT/'_shared_cache'/'ussd_compare16_v1'
CACHE_POINTER=RUN/'02_audio_feature_cache.json'
EXPECTED_CONFIG_SHA='4baf8b75324db30e632a2935ab469058d54cd1c7b1806d5cc220dc7b124c454c'

def valid_compare16_cache(root):
    root=Path(root)
    need=[
        root/'participant_manifest.csv',
        root/'preprocessing_train.json',
        root/'preprocessing_dev.json',
        root/'participant_features',
    ]
    if not all(p.exists() for p in need):
        return False,None

    try:
        man=pd.read_csv(root/'participant_manifest.csv')
        tr=json.loads((root/'preprocessing_train.json').read_text())
        dv=json.loads((root/'preprocessing_dev.json').read_text())
    except Exception:
        return False,None

    if 'split' not in man.columns or 'participant_id' not in man.columns:
        return False,None

    split=man['split'].astype(str).str.lower()
    train_ids=set(man.loc[split.eq('train'),'participant_id'].astype(int))
    dev_ids=set(man.loc[split.eq('dev'),'participant_id'].astype(int))

    ok=(
        len(train_ids)==107 and
        len(dev_ids)==34 and
        440 not in dev_ids and
        not (train_ids & dev_ids) and
        int(tr.get('prepared',-1))==107 and
        int(dv.get('prepared',-1))==34 and
        tr.get('compare16_config_sha256')==EXPECTED_CONFIG_SHA and
        dv.get('compare16_config_sha256')==EXPECTED_CONFIG_SHA and
        tr.get('test_opened') is False and
        dv.get('test_opened') is False
    )
    info={
        'path':str(root),
        'train_participants':len(train_ids),
        'dev_participants':len(dev_ids),
        'participant_440_excluded':440 not in dev_ids,
        'compare16_config_sha256':tr.get('compare16_config_sha256'),
        'test_opened':bool(tr.get('test_opened') or dv.get('test_opened')),
    }
    return ok,info

# Prefer the already verified historical canonical cache, then our new shared cache.
FEATURES=None; cache_info=None; cache_source=None
for candidate,label in [
    (CANONICAL_CACHE,'existing canonical ussd_compare16 cache'),
    (SHARED_CACHE,'exact-v3 shared cache'),
]:
    ok,info=valid_compare16_cache(candidate)
    if ok:
        FEATURES=Path(candidate); cache_info=info; cache_source=label; break

if FEATURES is None:
    FEATURES=SHARED_CACHE
    FEATURES.mkdir(parents=True,exist_ok=True)

    cmd=[sys.executable,'-m','scripts.teachers.ussd_audio.prepare_compare16',
         '--daic-root',str(DAIC),'--output',str(FEATURES),'--split','both',
         '--smile-extract',str(SMILE),'--compare16-config',str(CONFIG)]

    print('No valid shared cache found. Building ONCE at:',FEATURES)
    proc=subprocess.run(cmd,cwd=CODE,text=True,capture_output=True)
    if proc.stdout: print(proc.stdout)
    if proc.stderr: print(proc.stderr,file=sys.stderr)
    if proc.returncode!=0:
        raise RuntimeError(
            f'prepare_compare16 failed with exit code {proc.returncode}. '
            'The complete child-process traceback is printed immediately above.'
        )

    ok,cache_info=valid_compare16_cache(FEATURES)
    assert ok,'New ComParE16 shared cache failed validation.'
    cache_source='new exact-v3 shared cache'

pointer={
    'cache_type':'USSD-compatible ComParE16 participant features',
    'cache_source':cache_source,
    **cache_info,
    'immutable_for_this_run':True,
    'run_id':RUN.name,
}
CACHE_POINTER.write_text(json.dumps(pointer,indent=2)+'\n')

print('REUSING FEATURE CACHE:',FEATURES)
print(json.dumps(pointer,indent=2))
print('No run-specific ComParE16 recalculation is needed.')

In [ ]:
TEACHER=RUN/'02_audio_teacher'
DEV_AUDIT=TEACHER/'dev_audit'
TEACHER.mkdir(parents=True,exist_ok=True)

subprocess.run([sys.executable,'-m','scripts.teachers.ussd_audio.audit_frozen',
                '--features',str(FEATURES),'--author-root',str(SRC),
                '--output',str(DEV_AUDIT)],cwd=CODE,check=True)

subprocess.run([sys.executable,'-m','scripts.teachers.ussd_audio.audit_train_crop',
                '--features',str(FEATURES),'--author-root',str(SRC),
                '--output',str(TEACHER)],cwd=CODE,check=True)

audit=json.loads((DEV_AUDIT/'audit.json').read_text())
m=audit['local_dev34']
print(json.dumps(m,indent=2))
assert audit['author_commit']==USSD_COMMIT
assert audit['checkpoint_sha256']=='497ba1396f2e424dcbd5bc1dcf1f6636649d4681b30903ff03f6f8cbc4869e56'
assert audit['normalization_sha256']=='44d0516929fe893c2020fbcb0b7d1e521ab94758286899e0151e27b4a2c7790d'
assert audit['compare16_config_sha256']=='4baf8b75324db30e632a2935ab469058d54cd1c7b1806d5cc220dc7b124c454c'
assert audit['test_opened'] is False
assert round(float(m['macro_f1']),4)==0.7875

# Teacher epoch 0 means the ORIGINAL released USSD run-4 checkpoint.
USSD_EXP='cnn_lstm_feature_compare16_delta_feat_dim_384_batch_20_lr_0.003_wdecay_0_lrf_2_alpha_4e-06'
orig_ckpt=SRC/'speaker_disentanglement/best_model'/USSD_EXP/'model/4/md_35_epochs.pth'
orig_stats=SRC/'speaker_disentanglement/best_model'/USSD_EXP/'model/4/data_saver.pickle'
assert orig_ckpt.is_file() and orig_stats.is_file()

shutil.copy2(orig_ckpt,TEACHER/'epoch0_original_teacher.pt')
shutil.copy2(orig_stats,TEACHER/'epoch0_original_normalization.pickle')
shutil.copy2(orig_ckpt,TEACHER/'selected_teacher.pt')
shutil.copy2(orig_stats,TEACHER/'selected_normalization.pickle')
shutil.copy2(TEACHER/'train_run4_crop_kd_targets.csv',TEACHER/'selected_train_kd_targets.csv')

dep_f1=float(m['classification_report']['1']['f1-score'])
auroc=float(m['auroc_soft_mean_probability'])
selection={
    'teacher':'USSD ComParE16 + LSTM released run #4',
    'epoch0_definition':'original released checkpoint before any local fine-tuning',
    'epoch0_source_commit':USSD_COMMIT,
    'epoch0_checkpoint':'md_35_epochs.pth',
    'epoch0_checkpoint_sha256':audit['checkpoint_sha256'],
    'epoch0_dev34':{
        'macro_f1':float(m['macro_f1']),
        'depressed_f1':dep_f1,
        'auroc':auroc,
        'confusion_matrix':m['confusion_matrix'],
    },
    'selection_rule':'start with epoch 0; replace only on strict lexicographic DEV improvement: macro-F1, depressed-F1, AUROC',
    'fine_tuning_attempted':False,
    'selected_epoch':0,
    'selected_source':'original released USSD checkpoint',
    'selected_targets':'selected_train_kd_targets.csv',
    'test_opened':False,
}
(TEACHER/'teacher_selection.json').write_text(json.dumps(selection,indent=2)+'\n')

print('PASS: USSD audio teacher DEV-34 macro-F1 =',m['macro_f1'])
print('SELECTED TEACHER: epoch 0 original released USSD checkpoint')